In [1]:
import os
import cdsapi
import datetime
import isodate
import math
import numpy as np
import pandas as pd
from pysolar.radiation import get_radiation_direct
from pysolar.solar import get_altitude
import pytz
import xarray
from scipy import __name__ as scipy_name

client = cdsapi.Client() # Making a connection to CDS, to fetch data.


# The fields to be fetched from the single-level source.
singlelevelfields = [
                        '10m_u_component_of_wind',
                        '10m_v_component_of_wind',
                        '2m_temperature',
                        'geopotential',
                        'land_sea_mask',
                        'mean_sea_level_pressure',
                        'toa_incident_solar_radiation',
                        'total_precipitation'
                    ]

# The fields to be fetched from the pressure-level source.
pressurelevelfields = [
                        'u_component_of_wind',
                        'v_component_of_wind',
                        'geopotential',
                        'specific_humidity',
                        'temperature',
                        'vertical_velocity'
                    ]

# The 13 pressure levels.
pressure_levels = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]

# Initializing other required constants.
pi = math.pi
gap = 6 # There is a gap of 6 hours between each graphcast prediction.
first_prediction = datetime.datetime(2022, 1, 1, 18, 0) # Timestamp of the first prediction.
watts_to_joules = 3600
predictions_steps = 12 # Predicting for 4 timestamps.
lat_range = range(-180, 181, 1) # Latitude range.
lon_range = range(0, 360, 1) # Longitude range.

smallModel = True

if smallModel == False:
    spatial_resolution = '0.25/0.25'
else:
    spatial_resolution = '1.0/1.0'


# A utility function used for ease of coding.
# Converting the variable to a datetime object.
def toDatetime(dt) -> datetime.datetime:
    if isinstance(dt, datetime.date) and isinstance(dt, datetime.datetime):
        return dt
    
    elif isinstance(dt, datetime.date) and not isinstance(dt, datetime.datetime):
        return datetime.datetime.combine(dt, datetime.datetime.min.time())
    
    elif isinstance(dt, str):
        if 'T' in dt:
            return isodate.parse_datetime(dt)
        else:
            return datetime.datetime.combine(isodate.parse_date(dt), datetime.datetime.min.time())
        
_SEC_PER_HOUR = 3600
_HOUR_PER_DAY = 24
SEC_PER_DAY = _SEC_PER_HOUR * _HOUR_PER_DAY
_AVG_DAY_PER_YEAR = 365.24219




def get_year_progress(seconds_since_epoch: np.ndarray) -> np.ndarray:
  """Computes year progress for times in seconds.

  Args:
    seconds_since_epoch: Times in seconds since the "epoch" (the point at which
      UNIX time starts).

  Returns:
    Year progress normalized to be in the [0, 1) interval for each time point.
  """

  # Start with the pure integer division, and then float at the very end.
  # We will try to keep as much precision as possible.
  years_since_epoch = (
      seconds_since_epoch / SEC_PER_DAY / np.float64(_AVG_DAY_PER_YEAR)
  )
  # Note depending on how these ops are down, we may end up with a "weak_type"
  # which can cause issues in subtle ways, and hard to track here.
  # In any case, casting to float32 should get rid of the weak type.
  # [0, 1.) Interval.
  return np.mod(years_since_epoch, 1.0).astype(np.float32)


def get_day_progress(
    seconds_since_epoch: np.ndarray,
    longitude: np.ndarray,
) -> np.ndarray:
  """Computes day progress for times in seconds at each longitude.

  Args:
    seconds_since_epoch: 1D array of times in seconds since the 'epoch' (the
      point at which UNIX time starts).
    longitude: 1D array of longitudes at which day progress is computed.

  Returns:
    2D array of day progress values normalized to be in the [0, 1) inverval
      for each time point at each longitude.
  """

  # [0.0, 1.0) Interval.
  day_progress_greenwich = (
      np.mod(seconds_since_epoch, SEC_PER_DAY) / SEC_PER_DAY
  )

  # Offset the day progress to the longitude of each point on Earth.
  longitude_offsets = np.deg2rad(longitude) / (2 * np.pi)
  day_progress = np.mod(
      day_progress_greenwich[..., np.newaxis] + longitude_offsets, 1.0
  )
  return day_progress.astype(np.float32)

2025-04-14 01:12:37,179 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-04-14 01:12:37,180 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.


In [2]:
from tqdm import tqdm

In [4]:
data_file = 'inputs_data.pkl' 

# Function to check for file existence and load or retrieve data
def load_or_retrieve_data():
    if os.path.exists(data_file):
        print("Loading data from file...")
        values = pd.read_pickle(data_file)
    else:
        print("File not found, retrieving data...")
        single, pressure = getSingleAndPressureValues()
        inputs = pd.merge(pressure, single, left_index=True, right_index=True, how='inner')
        inputs = integrateProgress(inputs)
        inputs = formatData(inputs)
        values = {'inputs': inputs}  # Store the DataFrame in a dictionary
        pd.to_pickle(values, data_file)  # Save to file for future runs
        print("Data saved to file.")
    return values

# Getting the single and pressure level values.
def getSingleAndPressureValues():
    client.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': singlelevelfields,
            'grid': spatial_resolution,
            'year': [2022],
            'month': [1],
            'day': [1],
            'time': ['00:00', '01:00', '02:00', '03:00', '04:00', '05:00', '06:00', '07:00', '08:00', '09:00', '10:00', '11:00', '12:00'],
            'format': 'netcdf'
        },
        'single-level.nc'
    )
    singlelevel = xarray.open_dataset('single-level.nc', engine=scipy_name).to_dataframe()
    singlelevel = singlelevel.rename(columns={col: singlelevelfields[ind] for ind, col in enumerate(singlelevel.columns.values.tolist())})
    singlelevel = singlelevel.rename(columns={'geopotential': 'geopotential_at_surface'})

    # Calculating the sum of the last 6 hours of rainfall.
    singlelevel = singlelevel.sort_index()
    singlelevel['total_precipitation_6hr'] = singlelevel.groupby(level=[0, 1])['total_precipitation'].rolling(window=6, min_periods=1).sum().reset_index(level=[0, 1], drop=True)
    singlelevel.pop('total_precipitation')

    client.retrieve(
        'reanalysis-era5-pressure-levels',
        {
            'product_type': 'reanalysis',
            'variable': pressurelevelfields,
            'grid': spatial_resolution,
            'year': [2022],
            'month': [1],
            'day': [1],
            'time': ['06:00', '12:00'],
            'pressure_level': pressure_levels,
            'format': 'netcdf'
        },
        'pressure-level.nc'
    )
    pressurelevel = xarray.open_dataset('pressure-level.nc', engine=scipy_name).to_dataframe()
    pressurelevel = pressurelevel.rename(columns={col: pressurelevelfields[ind] for ind, col in enumerate(pressurelevel.columns.values.tolist())})

    return singlelevel, pressurelevel

# Adding sin and cos of the year progress.
def addYearProgress(secs, data):
    progress = get_year_progress(secs)
    data['year_progress_sin'] = math.sin(2 * math.pi * progress)
    data['year_progress_cos'] = math.cos(2 * math.pi * progress)
    return data

# Adding sin and cos of the day progress.
def addDayProgress(secs, lon: str, data: pd.DataFrame):
    lons = data.index.get_level_values(lon).unique()
    progress: np.ndarray = get_day_progress(secs, np.array(lons))
    prxlon = {lon: prog for lon, prog in list(zip(list(lons), progress.tolist()))}
    data['day_progress_sin'] = data.index.get_level_values(lon).map(lambda x: math.sin(2 * math.pi * prxlon[x]))
    data['day_progress_cos'] = data.index.get_level_values(lon).map(lambda x: math.cos(2 * math.pi * prxlon[x]))
    return data

# Adding day and year progress.
def integrateProgress(data: pd.DataFrame):
    for dt in data.index.get_level_values('time').unique():
        seconds_since_epoch = toDatetime(dt).timestamp()
        data = addYearProgress(seconds_since_epoch, data)
        data = addDayProgress(seconds_since_epoch, 'longitude' if 'longitude' in data.index.names else 'lon', data)
    return data

# Adding batch field and renaming some others.
def formatData(data: pd.DataFrame) -> pd.DataFrame:
    data = data.rename_axis(index={'latitude': 'lat', 'longitude': 'lon'})
    if 'batch' not in data.index.names:
        data['batch'] = 0
        data = data.set_index('batch', append=True)
    return data

# Main function to run the process

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

values = load_or_retrieve_data()
print("Data processing complete and saved.")

File not found, retrieving data...


2025-04-14 01:21:11,227 INFO Request ID is 6057d8e8-8c1b-42b6-bd51-44a759afd361
2025-04-14 01:21:11,728 INFO status has been updated to accepted
2025-04-14 01:21:20,700 INFO status has been updated to running
2025-04-14 01:21:25,965 INFO status has been updated to successful


SSLError: HTTPSConnectionPool(host='object-store.os-api.cci2.ecmwf.int', port=443): Max retries exceeded with url: /cci2-prod-cache-2/2025-04-13/835661177b650ef64924f538a27f109d.zip (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))

In [5]:
import cdsapi

client = cdsapi.Client()

dataset = 'reanalysis-era5-pressure-levels'
request = {
  'product_type': ['reanalysis'],
  'variable': ['geopotential'],
  'year': ['2024'],
  'month': ['03'],
  'day': ['01'],
  'time': ['13:00'],
  'pressure_level': ['1000'],
  'data_format': 'grib',
}
target = 'download.grib'

client.retrieve(dataset, request, target)

2025-04-14 01:22:18,493 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-04-14 01:22:18,494 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
2025-04-14 01:22:18,934 INFO Request ID is 264c17c0-d2fd-4795-af3e-d9e7171d041a
2025-04-14 01:22:19,148 INFO status has been updated to accepted
2025-04-14 01:22:28,287 INFO status has been updated to running
2025-04-14 01:22:33,548 INFO status has been updated to successful


SSLError: HTTPSConnectionPool(host='object-store.os-api.cci2.ecmwf.int', port=443): Max retries exceeded with url: /cci2-prod-cache-2/2025-04-07/79e8588ed2d6bf970f3f56839d004865.grib (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))